# Tema 1 - Introduccion a la Teoria de Maquinas y Mecanismos

**Teoria de Maquinas y Mecanismos - 2o GIERM**

---

## Objetivos de aprendizaje

- Distinguir entre maquina, mecanismo y cadena cinematica
- Clasificar mecanismos elementales (levas, engranajes, barras, tornillo, flexibles, hidraulicos)
- Identificar y clasificar eslabones y pares cinematicos
- Esquematizar mecanismos usando simbolos normalizados (ISO 3952)
- Calcular grados de libertad con la **ecuacion de Gruebler**: $G = 3(N-1) - 2g - h$
- Aplicar equivalencias cinematicas y reconocer inversiones de un mecanismo
- Evaluar el angulo de transmision y aplicar la **ley de Grashof**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Arc, Circle, Rectangle, Polygon
from matplotlib.lines import Line2D

# --- Paleta de colores estandar ---
COLOR_PRINCIPAL = '#2171b5'
COLOR_FIJO      = '#636363'
COLOR_PUNTO     = '#238b45'
COLOR_ROJO      = '#cb181d'
COLOR_NARANJA   = '#ff7f00'
COLOR_MORADO    = '#6a3d9a'
COLOR_CLARO     = '#a6cee3'

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.grid': True,
    'axes.grid.which': 'major',
    'grid.alpha': 0.3,
    'lines.linewidth': 1.5,
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.figsize': (12, 6),
})

# --- Helpers de dibujo para mecanismos ---
def draw_link(ax, p1, p2, color=COLOR_PRINCIPAL, lw=5, zorder=2):
    ax.plot([p1[0], p2[0]], [p1[1], p2[1]], '-', color=color, lw=lw,
            solid_capstyle='round', zorder=zorder)

def draw_joint(ax, p, fixed=False, color='black', ms=10, zorder=5):
    if fixed:
        ax.plot(p[0], p[1], 's', color=color, ms=ms, zorder=zorder,
                markeredgecolor='black', markeredgewidth=1.5)
    else:
        ax.plot(p[0], p[1], 'o', color='white', ms=ms, zorder=zorder,
                markeredgecolor='black', markeredgewidth=2)

def draw_ground(ax, center, width, y_offset=-0.15):
    x0 = center[0] - width/2
    y0 = center[1] + y_offset
    n_hash = 8
    for i in range(n_hash + 1):
        xi = x0 + i * width / n_hash
        ax.plot([xi, xi - 0.08], [y0, y0 - 0.12], 'k-', lw=0.8, zorder=1)
    ax.plot([x0, x0 + width], [y0, y0], 'k-', lw=1.5, zorder=1)

def draw_slider(ax, p, angle=0, size=0.3, color=COLOR_FIJO):
    rect = Rectangle((p[0]-size/2, p[1]-size/4), size, size/2,
                      angle=angle, facecolor=color, edgecolor='black',
                      lw=1.5, alpha=0.3, zorder=3)
    ax.add_patch(rect)
    ax.plot(p[0], p[1], 'o', color='white', ms=8,
            markeredgecolor='black', markeredgewidth=2, zorder=5)

print('Configuracion y helpers de dibujo cargados.')

---

## 1. Formulario del Tema

| Concepto | Formula | Notas |
|----------|---------|-------|
| **Ecuacion de Gruebler** | $\boxed{G = 3(N-1) - 2g - h}$ | $N$ = eslabones, $g$ = pares clase I, $h$ = pares clase II |
| **Par clase I (tipo g)** | Restringe 2 GDL | Rotacion, prismatico, rodadura sin deslizamiento |
| **Par clase II (tipo h)** | Restringe 1 GDL | Rodadura con deslizamiento (leva) |
| **Par n-ario** | Equivale a $n-1$ pares binarios | Un par ternario = 2 pares binarios |
| **Ley de Grashof** | $\boxed{s + l \leq p + q}$ | $s$ = mas corta, $l$ = mas larga, $p, q$ = restantes |
| **Angulo de transmision** | $\boxed{\mu = \angle(\text{biela}, \text{salida})}$ | Optimo: $\mu = 90°$. Minimo aceptable: $\mu > 40°$ |
| **GDL de un solido libre** | Plano: 3 GDL, Espacio: 6 GDL | $(x, y, \theta)$ en el plano |
| **Condicion de mecanismo** | $G > 0$ | $G = 0$: estructura; $G < 0$: hiperestatico |

### Tabla de pares cinematicos planos

| Par | Clase | Mov. permitido | Mov. restringido | Simbolo |
|-----|-------|----------------|------------------|---------|
| **Rotacion** (articulacion) | I ($g$) | 1 rotacion | 2 traslaciones | $\bigcirc$ |
| **Prismatico** (corredera) | I ($g$) | 1 traslacion | 1 traslacion + 1 rotacion | $\square$ |
| **Rodadura sin deslizamiento** | I ($g$) | 1 rotacion | 2 traslaciones | Circulos tangentes |
| **Rodadura con deslizamiento** (leva) | II ($h$) | 1 rotacion + 1 traslacion | 1 traslacion | Perfiles tangentes |

---

## 2. Definiciones fundamentales

### 2.1 Maquina

> **Maquina:** combinacion de solidos resistentes, agrupados y conectados de tal modo que son capaces de **transmitir una fuerza o un momento** desde una fuente de energia hasta un punto de consumo, donde se vence una resistencia, realizando para ello un **trabajo**.

Subsistemas de una maquina:
- **Sistema motriz:** proporciona la energia (motor)
- **Sistema transmisor:** transmite el movimiento (mecanismo)
- **Sistema receptor:** recibe el trabajo util (herramienta, rueda)
- **Sistemas auxiliares:** lubricacion, refrigeracion, sustentacion

### 2.2 Cadena cinematica

> **Cadena cinematica:** sistema formado por varios cuerpos conectados entre si con capacidad de **movimiento relativo** entre ellos.

### 2.3 Mecanismo

> **Mecanismo:** cadena cinematica en la que **fijamos el movimiento de uno de los cuerpos** (bastidor o barra fija).

### Diferencia clave

| Concepto | Asociado a | Ejemplo |
|----------|-----------|---------||
| **Mecanismo** | Movimientos | Piston-biela-manivela transmite movimiento |
| **Maquina** | Trabajo | Motor de combustion realiza trabajo |

$$\text{Mecanismo} \subset \text{Maquina}$$

In [ ]:
# Diagrama de bloques: estructura de una maquina
fig, ax = plt.subplots(figsize=(12, 6))
ax.set_xlim(0, 12)
ax.set_ylim(0, 8)
ax.axis('off')
ax.set_title('Esquema general de una maquina', fontsize=14, fontweight='bold')

bloques = [
    (1.0, 3.5, 2.2, 1.2, 'Sistema\nmotriz',     COLOR_ROJO),
    (4.5, 3.5, 2.8, 1.2, 'Sistema\ntransmisor',  COLOR_PRINCIPAL),
    (8.5, 3.5, 2.2, 1.2, 'Sistema\nreceptor',    COLOR_PUNTO),
    (4.5, 6.0, 2.8, 1.2, 'Sistemas\nauxiliares',  COLOR_NARANJA),
    (4.5, 1.0, 2.8, 1.2, 'Sistemas de\nsustentacion', COLOR_FIJO),
]

for x, y, w, h, label, color in bloques:
    rect = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.1',
                           facecolor=color, edgecolor='black', lw=2, alpha=0.2)
    ax.add_patch(rect)
    ax.text(x + w/2, y + h/2, label, ha='center', va='center',
            fontsize=11, fontweight='bold', color='black')

# Flechas horizontales
arrow_kw = dict(arrowstyle='->', color='black', lw=2.5)
ax.annotate('', xy=(4.4, 4.1), xytext=(3.3, 4.1), arrowprops=arrow_kw)
ax.annotate('', xy=(8.4, 4.1), xytext=(7.4, 4.1), arrowprops=arrow_kw)

# Flechas verticales
ax.annotate('', xy=(5.9, 5.9), xytext=(5.9, 4.8), arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))
ax.annotate('', xy=(5.9, 3.4), xytext=(5.9, 2.3), arrowprops=dict(arrowstyle='<->', color='gray', lw=1.5))

ax.text(3.8, 4.5, 'Energia', fontsize=9, ha='center', fontstyle='italic', color=COLOR_ROJO)
ax.text(7.9, 4.5, 'Trabajo', fontsize=9, ha='center', fontstyle='italic', color=COLOR_PUNTO)

plt.tight_layout()
plt.show()

---

## 3. Tipos de mecanismos

### 3.1 Segun la geometria del movimiento

| Tipo | Descripcion | Ejemplo |
|------|-------------|---------||
| **Plano** | Todos los puntos se mueven en planos paralelos | Piston-biela-manivela |
| **Espacial** | Puntos se mueven en 3D | Brazo robotico |

### 3.2 Segun la topologia de la cadena

| Tipo | Descripcion | Ejemplo |
|------|-------------|---------||
| **Cerrado** | Cadena cinematica forma un lazo cerrado | Cuadrilatero articulado |
| **Abierto** | Cadena cinematica sin cerrar | Robot serial |

---

## 4. Mecanismos elementales

Los mecanismos suelen consistir en una combinacion apropiada de mecanismos elementales:

| Tipo | Descripcion | Transformacion de movimiento |
|------|-------------|-----------------------------||
| **a) Levas** | Leva (imprime movimiento) + seguidor | Rotacion $\to$ perfil programado |
| **b) Engranajes** | Ruedas dentadas acopladas | Rotacion $\to$ rotacion (ratio constante) |
| **c) Tornillo** | Par helicoidal | Rotacion $\to$ traslacion |
| **d) Elementos flexibles** | Correas, cadenas, resortes | Rotacion $\to$ rotacion (ejes distantes) |
| **e) Elementos fluidos** | Cilindros hidraulicos/neumaticos | Presion $\to$ traslacion |
| **f) Barras** | Solidos rigidos enlazados | Diversos (segun configuracion) |

### Mecanismos de barras fundamentales

**1) Cuadrilatero articulado (4 barras):** 3 solidos moviles + 1 fijo, unidos por 4 pares de rotacion.

**2) Biela-manivela-corredera:** Caso particular del cuadrilatero donde una barra se hace infinitamente larga y su movimiento rotativo degenera en una **traslacion**.

In [ ]:
# Diagramas: cuadrilatero articulado vs biela-manivela-corredera
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- a) Cuadrilatero articulado ---
ax = axes[0]
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('a) Cuadrilatero articulado (4 barras)', fontsize=12, fontweight='bold')

A0 = np.array([0.5, 0])
B0 = np.array([4.5, 0])
th2 = np.radians(60)
L2, L3, L4 = 1.5, 3.5, 2.5
A = A0 + L2 * np.array([np.cos(th2), np.sin(th2)])
B = np.array([3.8, 2.2])

draw_link(ax, A0, B0, color=COLOR_FIJO, lw=8)
draw_link(ax, A0, A, color=COLOR_ROJO)
draw_link(ax, A, B, color=COLOR_PRINCIPAL)
draw_link(ax, B0, B, color=COLOR_PUNTO)

draw_ground(ax, A0, 0.6)
draw_ground(ax, B0, 0.6)
draw_joint(ax, A0, fixed=True)
draw_joint(ax, B0, fixed=True)
draw_joint(ax, A)
draw_joint(ax, B)

ax.text(A0[0]-0.3, A0[1]-0.5, '$A_0$', fontsize=11, ha='center', fontweight='bold')
ax.text(B0[0]+0.3, B0[1]-0.5, '$B_0$', fontsize=11, ha='center', fontweight='bold')
ax.text(A[0]-0.3, A[1]+0.2, '$A$', fontsize=11, ha='center', fontweight='bold')
ax.text(B[0]+0.3, B[1]+0.2, '$B$', fontsize=11, ha='center', fontweight='bold')
ax.text(2.5, -0.7, '1 (fijo)', fontsize=10, ha='center', color=COLOR_FIJO)
ax.text(A0[0]-0.1, A[1]/2, '2', fontsize=10, ha='right', color=COLOR_ROJO)
ax.text((A[0]+B[0])/2, (A[1]+B[1])/2+0.3, '3', fontsize=10, ha='center', color=COLOR_PRINCIPAL)
ax.text(B0[0]+0.4, B[1]/2, '4', fontsize=10, ha='left', color=COLOR_PUNTO)

# --- b) Biela-manivela-corredera ---
ax = axes[1]
ax.set_xlim(-1, 7)
ax.set_ylim(-1.5, 3)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('b) Biela-manivela-corredera', fontsize=12, fontweight='bold')

O = np.array([0, 0])
th = np.radians(45)
r = 1.2
l_biela = 3.5
A_bm = O + r * np.array([np.cos(th), np.sin(th)])
x_piston = r * np.cos(th) + np.sqrt(l_biela**2 - (r*np.sin(th))**2)
P = np.array([x_piston, 0])

draw_link(ax, O, A_bm, color=COLOR_ROJO, lw=5)
draw_link(ax, A_bm, P, color=COLOR_PRINCIPAL, lw=5)

slider_rect = Rectangle((P[0]-0.25, P[1]-0.35), 0.5, 0.7,
                          facecolor=COLOR_PUNTO, alpha=0.3, edgecolor='black', lw=2)
ax.add_patch(slider_rect)

ax.plot([-0.5, 6], [0, 0], '--', color=COLOR_FIJO, lw=1, alpha=0.5)
draw_ground(ax, O, 0.6)
draw_joint(ax, O, fixed=True)
draw_joint(ax, A_bm)
ax.plot(P[0], P[1], 'o', color='white', ms=8, markeredgecolor='black', markeredgewidth=2, zorder=5)

ax.plot([P[0]-0.6, 6.5], [-0.4, -0.4], '-', color=COLOR_FIJO, lw=2)
ax.plot([P[0]-0.6, 6.5], [0.4, 0.4], '-', color=COLOR_FIJO, lw=2)

ax.text(O[0]-0.4, O[1]-0.6, '$O$', fontsize=11, fontweight='bold')
ax.text(A_bm[0]-0.3, A_bm[1]+0.2, '$A$', fontsize=11, fontweight='bold')
ax.text(P[0]+0.5, P[1], '$P$', fontsize=11, fontweight='bold')
ax.text(0.2, 1.2, '2\n(manivela)', fontsize=9, color=COLOR_ROJO, ha='center')
ax.text((A_bm[0]+P[0])/2, (A_bm[1]+P[1])/2+0.3, '3 (biela)', fontsize=9, color=COLOR_PRINCIPAL)
ax.text(P[0], -1.0, '4 (corredera)', fontsize=9, ha='center', color=COLOR_PUNTO)

plt.tight_layout()
plt.show()

---

## 5. Eslabon o barra

Un **eslabon** o **barra** es cada uno de los elementos del mecanismo que tiene movimiento relativo respecto a los otros.

### Clasificacion por tipo de material

| Tipo | Ejemplo |
|------|---------|
| Rigido (o varios rigidamente unidos) | Biela de motor |
| Unirrigido | Correas |
| Flexible | Muelles, resortes |
| Fluido | Aceite hidraulico |

### Clasificacion por funcion

| Funcion | Nombre | Descripcion |
|---------|--------|-------------|
| Entrada | Impulsor | Recibe el movimiento del motor |
| Transmision | Acoplador / Biela | Transmite el movimiento |
| Salida | Conducido | Realiza el trabajo util |

### Clasificacion por movimiento

| Tipo | Movimiento | Ejemplo en biela-manivela |
|------|-----------|---------------------------|
| **Manivela** | Giro continuo alrededor de un punto fijo | Eslabon de entrada (2) |
| **Balancin** | Giro alternativo alrededor de un punto fijo | Eslabon de salida en manivela-balancin |
| **Corredera** | Traslacion a lo largo de una linea | Piston (4) |
| **Biela** | Rotacion + traslacion combinadas | Eslabon acoplador (3) |
| **Barra fija** | Sin movimiento | Bastidor (1) |

### Clasificacion por numero de conexiones

| Tipo | Conectado a | Ejemplo |
|------|------------|---------||
| **Binario** | 2 eslabones | La mayoria de barras |
| **Ternario** | 3 eslabones | Barra con 3 articulaciones |
| **n-ario** | $n$ eslabones | Nudo central complejo |

---

## 6. Par cinematico

> Un **par cinematico** es la union entre dos barras de un mecanismo con capacidad de **movimiento relativo** entre ellas.

### 6.1 Clasificacion por movimiento relativo (general, 3D)

| # | Par | Mov. permitido | GDL relativos |
|---|-----|----------------|---------------|
| 1 | **Rotacion** (articulacion) | Rotacion alrededor de un eje | 1 |
| 2 | **Prismatico** | Traslacion en una direccion | 1 |
| 3 | **Cilindrico** | Rotacion + traslacion independientes | 2 |
| 4 | **Helicoidal** (tornillo) | Rotacion + traslacion dependientes | 1 |
| 5 | **Rodadura sin deslizamiento** | Rotacion relativa | 1 |
| 6 | **Leva** (rodadura con deslizamiento) | Rotacion + deslizamiento | 2 |
| 7 | **Esferico** (rotula) | Rotacion en cualquier direccion | 3 |
| 8 | **Plano** | 2 traslaciones + 1 rotacion | 3 |

### 6.2 Pares cinematicos en mecanismos planos

En mecanismos planos solo aparecen 4 tipos:

| Par | Clase Gruebler | GDL restringidos | GDL permitidos |
|-----|----------------|------------------|----------------|
| **Rotacion** | I ($g$) | 2 | 1 (rotacion) |
| **Prismatico** | I ($g$) | 2 | 1 (traslacion) |
| **Rodadura sin deslizamiento** | I ($g$) | 2 | 1 (rotacion) |
| **Rodadura con deslizamiento** (leva) | II ($h$) | 1 | 2 (rotacion + deslizamiento) |

### 6.3 Clasificacion por eslabones que conectan

| Tipo | Conecta | Equivalencia para Gruebler |
|------|---------|----------------------------|
| **Binario** | 2 eslabones | 1 par |
| **Ternario** | 3 eslabones | 2 pares binarios |
| **n-ario** | $n$ eslabones | $n-1$ pares binarios |

> **Clave:** Un par ternario (punto donde se unen 3 barras) cuenta como **2 pares de clase I** en Gruebler, no como 1.

In [ ]:
# Diagrama: los 4 pares cinematicos planos
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ['Par de rotacion\n(Clase I)', 'Par prismatico\n(Clase I)',
          'Rodadura sin desl.\n(Clase I)', 'Rodadura con desl.\n(Clase II - Leva)']

for ax, title in zip(axes, titles):
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=10, fontweight='bold')

# a) Par de rotacion
ax = axes[0]
draw_link(ax, (-1.5, -0.5), (0, 0), color=COLOR_PRINCIPAL)
draw_link(ax, (0, 0), (1.5, 0.8), color=COLOR_ROJO)
draw_joint(ax, (0, 0))
ax.annotate('', xy=(0.7, 1.0), xytext=(1.0, 0.3),
            arrowprops=dict(arrowstyle='->', color=COLOR_NARANJA, lw=2, connectionstyle='arc3,rad=0.5'))
ax.text(1.2, 0.9, r'$\theta$', fontsize=14, color=COLOR_NARANJA)

# b) Par prismatico
ax = axes[1]
ax.add_patch(Rectangle((-1.5, -0.3), 3.0, 0.6, facecolor=COLOR_CLARO, edgecolor='black', lw=1.5))
ax.add_patch(Rectangle((-0.4, -0.5), 0.8, 1.0, facecolor=COLOR_PRINCIPAL, edgecolor='black', lw=2, alpha=0.5))
ax.annotate('', xy=(1.2, 0), xytext=(-0.8, 0),
            arrowprops=dict(arrowstyle='->', color=COLOR_NARANJA, lw=2.5))
ax.text(0.2, -1.2, '$d$', fontsize=14, color=COLOR_NARANJA, ha='center')

# c) Rodadura sin deslizamiento
ax = axes[2]
c1 = plt.Circle((0, 0.8), 0.8, fill=False, edgecolor=COLOR_PRINCIPAL, lw=2.5)
c2 = plt.Circle((0, -0.8), 0.8, fill=False, edgecolor=COLOR_ROJO, lw=2.5)
ax.add_patch(c1)
ax.add_patch(c2)
ax.plot(0, 0, 'ko', ms=6, zorder=5)
ax.text(0, 0.8, '+', fontsize=12, ha='center', va='center', color=COLOR_PRINCIPAL)
ax.text(0, -0.8, '+', fontsize=12, ha='center', va='center', color=COLOR_ROJO)
ax.annotate('', xy=(0.6, 1.3), xytext=(0.1, 1.5),
            arrowprops=dict(arrowstyle='->', color=COLOR_NARANJA, lw=2, connectionstyle='arc3,rad=-0.3'))

# d) Rodadura con deslizamiento (leva)
ax = axes[3]
theta_leva = np.linspace(0, 2*np.pi, 200)
r_leva = 0.7 + 0.3*np.cos(2*theta_leva)
x_leva = r_leva * np.cos(theta_leva)
y_leva = r_leva * np.sin(theta_leva) - 0.5
ax.plot(x_leva, y_leva, color=COLOR_PRINCIPAL, lw=2.5)
ax.plot(0, -0.5, '+', color=COLOR_PRINCIPAL, ms=10, mew=2)
ax.plot([0, 0], [0.3, 1.8], color=COLOR_ROJO, lw=3)
ax.add_patch(plt.Circle((0, 0.3), 0.15, fill=False, edgecolor=COLOR_ROJO, lw=2))
ax.annotate('', xy=(0, 1.5), xytext=(0, 0.8),
            arrowprops=dict(arrowstyle='->', color=COLOR_NARANJA, lw=2))

plt.tight_layout()
plt.show()

---

## 7. Esquematizacion y normalizacion

**Objetivo:** Obtener un esquema simplificado del mecanismo que sea facil de visualizar y permita un analisis sencillo (norma **UNE-EN ISO 3952**).

### Simbolos normalizados principales

| Designacion | Descripcion |
|-------------|-------------|
| **Barra** | Linea recta gruesa |
| **Eslabon fijo** (soporte) | Linea con trazos diagonales (hatching) |
| **Par de rotacion** | Circulo pequeno |
| **Par de rotacion unido al fijo** | Triangulo con circulo |
| **Par de rotacion en parte del eslabon** | Circulo en medio de barra |
| **Corredera sin par de rotacion** | Rectangulo |
| **Corredera con par de rotacion** | Rectangulo con circulo |
| **Eslabon ternario** | Triangulo con 3 circulos |

### Nomenclatura

- **Barras:** numeros ($1, 2, 3, ...$). Siempre $1$ = barra fija, $2$ = entrada
- **Pares:** letras mayusculas ($A, B, C, ...$). Subindice $0$ para pares con la barra fija: $A_0, B_0, ...$

---

## 8. Grados de libertad y ecuacion de Gruebler

### 8.1 Definicion

El numero de **grados de libertad (GDL)**, $G$, de un mecanismo es el numero de variables independientes que es necesario especificar para conocer en cualquier instante la posicion de todos sus eslabones.

- Solido rigido libre en el **espacio**: $G = 6$ ($x, y, z, \phi, \theta, \psi$)
- Solido rigido libre en el **plano**: $G = 3$ ($x, y, \theta$)

### 8.2 Ecuacion de Gruebler (mecanismos planos)

$$\boxed{G = 3(N - 1) - 2g - h}$$

Donde:
- $N$ = numero total de eslabones (incluyendo la barra fija)
- $g$ = numero de pares de **clase I** (restringen 2 GDL): rotacion, prismatico, rodadura sin deslizamiento
- $h$ = numero de pares de **clase II** (restringen 1 GDL): rodadura con deslizamiento (leva)

> **Importante:** Se resta 1 a $N$ porque la barra fija no aporta GDL. Cada par de clase I elimina 2 GDL del total y cada par de clase II elimina 1 GDL.

### 8.3 Interpretacion del resultado

| Valor de $G$ | Significado |
|---------------|------------|
| $G > 0$ | **Mecanismo** con $G$ grados de libertad (necesita $G$ actuadores) |
| $G = 0$ | **Estructura** (rigida, sin movimiento) |
| $G < 0$ | **Estructura hiperestatica** (restricciones redundantes) |

### 8.4 Ejemplo: cuadrilatero articulado

$N = 4$ eslabones (1 fijo + 3 moviles), $g = 4$ pares de rotacion, $h = 0$

$$G = 3(4 - 1) - 2 \times 4 - 0 = 9 - 8 = \boxed{1}$$

Un solo GDL: basta especificar $\theta_2$ (angulo de la manivela) para conocer la posicion completa.

### 8.5 Ejemplo: excavadora

$N = 12$ eslabones, $g = 15$ pares de rotacion, $h = 0$

$$G = 3(12 - 1) - 2 \times 15 = 33 - 30 = \boxed{3}$$

Necesita 3 actuadores (3 cilindros hidraulicos) para controlar brazo, antebrazo y cuchara.

In [ ]:
# Calculo automatico de GDL con Gruebler
def gruebler(N, g, h=0):
    return 3 * (N - 1) - 2 * g - h

ejemplos = [
    ('Cuadrilatero articulado',  4,  4, 0),
    ('Biela-manivela-corredera', 4,  4, 0),
    ('Camion volquete',          6,  7, 0),
    ('Excavadora',              12, 15, 0),
    ('Leva con rodillo',         4,  3, 1),
]

print('Ecuacion de Gruebler: G = 3(N-1) - 2g - h')
print('=' * 55)
print(f'{"Mecanismo":<30} {"N":>3} {"g":>3} {"h":>3} {"G":>4}')
print('-' * 55)
for nombre, N, g, h in ejemplos:
    G = gruebler(N, g, h)
    tipo = 'Mec.' if G > 0 else ('Estr.' if G == 0 else 'Hiper.')
    print(f'{nombre:<30} {N:>3} {g:>3} {h:>3} {G:>4}  ({tipo})')

---

### 8.6 Consideraciones y limitaciones de Gruebler

La ecuacion de Gruebler es sencilla pero tiene limitaciones:

**a) Mecanismos con el G predicho pero sin transmision del movimiento:**

Puede ocurrir que Gruebler de $G = 1$ pero el movimiento no se transmita a todos los eslabones.

**b) Grados de libertad pasivos:**

Los **rodillos** en mecanismos de leva giran libremente sin afectar al movimiento util. Gruebler da $G = 2$ pero el GDL real es 1 (el extra es pasivo).

**Ejemplo:** Leva con rodillo: $N = 4$, $g = 3$, $h = 1$

$$G = 3(4-1) - 2 \times 3 - 1 \times 1 = 9 - 6 - 1 = \boxed{2}$$

Pero el GDL real es 1 (el giro del rodillo es pasivo).

**c) Mecanismos con eslabones o pares redundantes:**

Restricciones geometricas especiales (simetrias, paralelismos) pueden hacer que Gruebler subestime los GDL reales.

> **Regla practica:** Gruebler da el resultado correcto en la gran mayoria de los casos. Cuando falla, hay que analizar si hay GDL pasivos o restricciones redundantes.

---

## 9. Equivalencia cinematica

Dos mecanismos son **cinematicamente equivalentes** si producen el mismo movimiento relativo entre sus eslabones, aunque su estructura fisica sea diferente.

### Equivalencias utiles

| Par original | Par equivalente | Condicion |
|-------------|-----------------|----------|
| Par de **tornillo** (helicoidal) | Par **prismatico** (clase I) | Los 2 movimientos son dependientes |
| Par de **leva** (clase II, $h$) | Par de **rotacion** + **prismatico** (2 pares clase I, $g$) | Anadir eslabon intermedio |
| **Engranaje** (rodadura sin desl.) | Par de **rotacion** (clase I) | Los centros de giro estan fijos |

> **Aplicacion:** Si un mecanismo tiene un par de leva ($h$), podemos sustituirlo por un eslabon adicional con 2 pares de rotacion. Esto cambia $N$ y $g$ pero el resultado de Gruebler debe ser el mismo.

---

## 10. Inversiones de un mecanismo

Una **inversion** consiste en fijar un eslabon diferente como bastidor, manteniendo las mismas longitudes y conexiones. Un mecanismo de $N$ eslabones tiene **$N$ inversiones posibles**.

### Inversiones del mecanismo biela-manivela (4 barras con corredera)

| Inversion | Eslabon fijo | Nombre del mecanismo resultante |
|-----------|-------------|-------------------------------|
| 1a | Bastidor original (1) | **Biela-manivela** clasico (motor de piston) |
| 2a | Manivela (2) | **Mecanismo de Whitworth** (retorno rapido) |
| 3a | Biela (3) | **Mecanismo de arrastre** |
| 4a | Corredera (4) | **Manivela de cilindro oscilante** |

> **Clave:** Todas las inversiones tienen los **mismos GDL** ($G = 1$) porque las longitudes y tipos de pares no cambian, solo cambia cual eslabon es fijo.

---

## 11. Angulo de transmision

El **angulo de transmision** $\mu$ es el angulo entre la biela (acoplador) y el eslabon de salida, medido en el punto de conexion entre ambos.

$$\boxed{\mu = \angle(\text{biela}, \text{eslabon de salida})}$$

### Importancia

- Mide la **eficiencia** de la transmision de fuerza/movimiento
- **Optimo:** $\mu = 90°$ (toda la fuerza de la biela se transmite al eslabon de salida)
- **Minimo aceptable:** $\mu > 40°$ (por debajo, la transmision es pobre y hay riesgo de agarrotamiento)
- $\mu = 0°$ o $\mu = 180°$: **punto muerto** (la fuerza de la biela no genera momento en la salida)

In [ ]:
# Visualizacion del angulo de transmision en un cuadrilatero articulado
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (th2_deg, titulo) in enumerate([(50, 'Buen angulo de transmision'), (160, 'Mal angulo de transmision')]):
    ax = axes[idx]
    ax.set_xlim(-1, 6)
    ax.set_ylim(-1.5, 4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(titulo, fontsize=12, fontweight='bold')

    a, b, c, d = 1.5, 3.5, 2.5, 4.0
    A0 = np.array([0, 0])
    B0 = np.array([d, 0])
    th2 = np.radians(th2_deg)
    A = A0 + a * np.array([np.cos(th2), np.sin(th2)])

    AB0 = np.linalg.norm(A - B0)
    cos_alpha = np.clip((c**2 + AB0**2 - b**2) / (2 * c * AB0), -1, 1)
    alpha = np.arccos(cos_alpha)
    angle_AB0 = np.arctan2(A[1] - B0[1], A[0] - B0[0])
    th4 = angle_AB0 + alpha
    B = B0 + c * np.array([np.cos(th4), np.sin(th4)])

    draw_link(ax, A0, B0, color=COLOR_FIJO, lw=7)
    draw_link(ax, A0, A, color=COLOR_ROJO)
    draw_link(ax, A, B, color=COLOR_PRINCIPAL)
    draw_link(ax, B0, B, color=COLOR_PUNTO)
    draw_ground(ax, A0, 0.5)
    draw_ground(ax, B0, 0.5)
    draw_joint(ax, A0, fixed=True)
    draw_joint(ax, B0, fixed=True)
    draw_joint(ax, A)
    draw_joint(ax, B)

    vec_BA = A - B
    vec_BB0 = B0 - B
    cos_mu = np.dot(vec_BA, vec_BB0) / (np.linalg.norm(vec_BA) * np.linalg.norm(vec_BB0))
    mu = np.degrees(np.arccos(np.clip(cos_mu, -1, 1)))

    ang1 = np.degrees(np.arctan2(vec_BB0[1], vec_BB0[0]))
    ang2 = np.degrees(np.arctan2(vec_BA[1], vec_BA[0]))
    if ang1 > ang2:
        ang1, ang2 = ang2, ang1
    arc = Arc(B, 0.8, 0.8, angle=0, theta1=ang1, theta2=ang2,
              color=COLOR_NARANJA, lw=2.5)
    ax.add_patch(arc)
    mid_ang = np.radians((ang1 + ang2) / 2)
    ax.text(B[0] + 0.6*np.cos(mid_ang), B[1] + 0.6*np.sin(mid_ang),
            f'$\\mu = {mu:.0f}\\degree$', fontsize=12, fontweight='bold',
            color=COLOR_NARANJA, ha='center', va='center')

    color_mu = COLOR_PUNTO if mu > 40 else COLOR_ROJO
    ax.text(2.5, -1.2, f'$\\mu = {mu:.1f}\\degree$  ' + ('(Bueno)' if mu > 40 else '(Malo)'),
            fontsize=12, ha='center', color=color_mu, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.3', facecolor=color_mu, alpha=0.15))

plt.tight_layout()
plt.show()

---

## 12. Ley de Grashof

Para un cuadrilatero articulado con eslabones de longitud $s$ (mas corta), $l$ (mas larga), $p$ y $q$ (las dos restantes):

$$\boxed{s + l \leq p + q}$$

### Clasificacion segun Grashof

| Condicion | Eslabon fijo | Tipo de mecanismo |
|-----------|-------------|-------------------|
| $s + l < p + q$ y fijamos el **adyacente a $s$** | Adyacente al mas corto | **Manivela-balancin** |
| $s + l < p + q$ y fijamos el **opuesto a $s$** | Opuesto al mas corto | **Doble manivela** |
| $s + l < p + q$ y fijamos **$s$** | El mas corto | **Doble balancin** |
| $s + l = p + q$ | Cualquiera | **Punto de cambio** |
| $s + l > p + q$ | Cualquiera | **Triple balancin** (ningun eslabon gira 360) |

In [ ]:
# Visualizacion: los 3 tipos de mecanismo de Grashof
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

configs = [
    ('Manivela-balancin', 1.0, 3.5, 2.5, 3.0, 'Fijamos adyacente a s'),
    ('Doble manivela',    1.0, 3.5, 2.5, 3.0, 'Fijamos opuesto a s'),
    ('Triple balancin',   2.0, 4.5, 2.0, 2.5, 's+l > p+q'),
]

for idx, (titulo, a, d, b_param, c, nota) in enumerate(configs):
    ax = axes[idx]
    ax.set_xlim(-2, 6)
    ax.set_ylim(-2, 4)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(titulo, fontsize=12, fontweight='bold')

    A0 = np.array([0, 0])
    B0 = np.array([d, 0]) if idx != 1 else np.array([c, 0])

    draw_link(ax, A0, B0, color=COLOR_FIJO, lw=7)
    draw_ground(ax, A0, 0.4)
    draw_ground(ax, B0, 0.4)

    n_pos = 6
    for i in range(n_pos):
        th = np.radians(30 + i * 300 / n_pos)
        L_input = a
        A = A0 + L_input * np.array([np.cos(th), np.sin(th)])
        alpha_val = 0.15 + 0.7 * (i == 0)
        ax.plot([A0[0], A[0]], [A0[1], A[1]], '-', color=COLOR_ROJO,
                lw=3, alpha=alpha_val)

        if i == 0:
            L_coupler = d if idx == 1 else b_param
            L_output = b_param if idx == 1 else c
            AB0_dist = np.linalg.norm(A - B0)
            if AB0_dist < abs(L_coupler - L_output) or AB0_dist > L_coupler + L_output:
                continue
            cos_a = np.clip((L_output**2 + AB0_dist**2 - L_coupler**2) / (2 * L_output * AB0_dist), -1, 1)
            ang = np.arccos(cos_a)
            ang_AB0 = np.arctan2(A[1]-B0[1], A[0]-B0[0])
            th4 = ang_AB0 + ang
            B = B0 + L_output * np.array([np.cos(th4), np.sin(th4)])

            draw_link(ax, A, B, color=COLOR_PRINCIPAL, lw=4)
            draw_link(ax, B0, B, color=COLOR_PUNTO, lw=4)
            draw_joint(ax, A)
            draw_joint(ax, B)
            draw_joint(ax, A0, fixed=True)
            draw_joint(ax, B0, fixed=True)

    barras = sorted([a, b_param, c, d])
    s_bar, l_bar = barras[0], barras[-1]
    p_bar, q_bar = barras[1], barras[2]
    cumple = s_bar + l_bar <= p_bar + q_bar
    ax.text(2.0, -1.5, nota, fontsize=9, ha='center', color=COLOR_FIJO, fontstyle='italic')
    ax.text(2.0, -1.9, f's+l={s_bar+l_bar:.1f} vs p+q={p_bar+q_bar:.1f}  '
            + ('SI' if cumple else 'NO'),
            fontsize=8, ha='center', color=COLOR_PUNTO if cumple else COLOR_ROJO)

plt.tight_layout()
plt.show()

---

## 13. Ejercicios resueltos

### Problema 1: GDL de mecanismo complejo (N=13)

**Enunciado:** Determinar los grados de libertad del mecanismo de la figura, que consta de una leva (disco grande), multiples barras articuladas, deslizaderas, un resorte, y un par polea-correa.

**Paso 1:** Identificar todos los eslabones.

Contamos cada cuerpo con movimiento independiente: eslabones 1 a 13.

$$\boxed{N = 13}$$

**Paso 2:** Identificar y clasificar todos los pares cinematicos.

Tras un conteo cuidadoso, incluyendo pares ternarios como $(n-1)$ pares binarios:

$$g = 17, \quad h = 0$$

**Paso 3:** Aplicar Gruebler.

$$G = 3(13 - 1) - 2 \times 17 = 36 - 34 = \boxed{2}$$

> **Clave:** El conteo de pares es la parte mas dificil. Hay que identificar cada par ternario como $n-1$ pares binarios y no olvidar los pares con la barra fija.

---

### Problema 2: GDL de mecanismo con disco y seguidor (N=14)

**Enunciado:** Determinar los GDL del mecanismo con $N = 14$ eslabones.

**Paso 1:** $N = 14$

**Paso 2:** Contar pares con cuidado (pares ternarios = $n-1$ binarios):

$$g = 19, \quad h = 0$$

**Paso 3:** Gruebler:

$$G = 3(14-1) - 2 \times 19 = 39 - 38 = \boxed{1}$$

> **Leccion:** Los pares ternarios cuentan como 2 pares binarios. Si olvidamos esto, el resultado sera incorrecto.

---

### Problema 3: Aplicar ley de Grashof

**Enunciado:** Cuadrilatero articulado con eslabones $a = 2$ cm, $b = 4$ cm, $c = 3$ cm, $d = 5$ cm. Determinar el tipo de mecanismo.

**Paso 1:** Ordenar: $s = 2$, $p = 3$, $q = 4$, $l = 5$.

**Paso 2:** Comprobar Grashof:

$$s + l = 2 + 5 = 7 \quad \text{vs} \quad p + q = 3 + 4 = 7$$

$$7 \leq 7 \implies \text{Se cumple Grashof (caso limite)}$$

**Paso 3:** Si fijamos el adyacente al mas corto:

$$\boxed{\text{Manivela-balancin (caso limite con punto de cambio)}}$$

---

## 14. Catalogo completo de ejercicios: todos los patrones

| # | Tipo de ejercicio | Formula clave | Dificultad |
|---|-------------------|---------------|------------|
| 1 | Calcular GDL dado un esquema | $G = 3(N-1) - 2g - h$ | Media |
| 2 | Clasificar pares cinematicos | Tabla de pares | Baja |
| 3 | Aplicar ley de Grashof | $s + l \leq p + q$ | Baja |
| 4 | Determinar tipo de mecanismo (Grashof) | Tabla manivela-balancin/doble manivela/... | Media |
| 5 | Calcular angulo de transmision | $\mu = \angle(\text{biela}, \text{salida})$ | Media |
| 6 | Identificar inversiones | Fijar cada eslabon | Baja |
| 7 | Esquematizar un mecanismo real | Norma ISO 3952 | Media |
| 8 | Equivalencia cinematica | Sustituir pares complejos | Alta |

### 14.1 Tipo 1: Calcular GDL dado un esquema

**Metodo sistematico:**

1. **Numerar todos los eslabones** (incluyendo la barra fija como eslabon 1)
2. **Identificar cada par cinematico** y marcarlo en el esquema
3. **Clasificar cada par:** clase I ($g$) o clase II ($h$)
4. **Pares n-arios:** cada par que conecta $n$ eslabones equivale a $n - 1$ pares binarios
5. **Aplicar Gruebler:** $G = 3(N-1) - 2g - h$

$$\boxed{G = 3(N-1) - 2g - h}$$

#### Ejercicio resuelto: Mecanismo de ventana (6 eslabones)

**Datos:** $N = 6$, $g = 7$, $h = 0$

**Paso 1:** Aplicar Gruebler:

$$G = 3(6 - 1) - 2 \times 7 = 15 - 14 = \boxed{1}$$

In [ ]:
# Tipo 5: Calcular angulo de transmision vs theta2
a, b, c, d = 1.5, 3.5, 2.5, 4.0

theta2_range = np.linspace(0, 2*np.pi, 500)
mu_values = []

for th2 in theta2_range:
    A0 = np.array([0, 0])
    B0 = np.array([d, 0])
    A = A0 + a * np.array([np.cos(th2), np.sin(th2)])

    AB0 = np.linalg.norm(A - B0)
    cos_alpha = np.clip((c**2 + AB0**2 - b**2) / (2 * c * AB0), -1, 1)
    alpha = np.arccos(cos_alpha)
    ang_AB0 = np.arctan2(A[1] - B0[1], A[0] - B0[0])
    th4 = ang_AB0 + alpha
    B = B0 + c * np.array([np.cos(th4), np.sin(th4)])

    vec_BA = A - B
    vec_BB0 = B0 - B
    cos_mu = np.dot(vec_BA, vec_BB0) / (np.linalg.norm(vec_BA) * np.linalg.norm(vec_BB0) + 1e-12)
    mu = np.degrees(np.arccos(np.clip(cos_mu, -1, 1)))
    mu_values.append(mu)

mu_values = np.array(mu_values)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(np.degrees(theta2_range), mu_values, color=COLOR_PRINCIPAL, lw=2.5, label=r'$\mu(\theta_2)$')
ax.axhline(90, color=COLOR_PUNTO, ls='--', lw=1.5, label=r'$\mu = 90°$ (optimo)')
ax.axhline(40, color=COLOR_ROJO, ls='--', lw=1.5, label=r'$\mu = 40°$ (minimo aceptable)')
ax.fill_between(np.degrees(theta2_range), 0, 40, alpha=0.1, color=COLOR_ROJO)

ax.set_xlabel(r'$\theta_2$ (grados)')
ax.set_ylabel(r'Angulo de transmision $\mu$ (grados)')
ax.set_title(f'Angulo de transmision vs posicion de entrada (a={a}, b={b}, c={c}, d={d})',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 360)
ax.set_ylim(0, 180)
ax.legend(fontsize=10)

mu_min = mu_values.min()
mu_max = mu_values.max()
ax.text(180, 160, f'$\\mu_{{min}} = {mu_min:.1f}°$, $\\mu_{{max}} = {mu_max:.1f}°$',
        fontsize=11, ha='center', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor=COLOR_CLARO, alpha=0.5))

plt.tight_layout()
plt.show()
print(f'Angulo de transmision minimo: {mu_min:.1f} grados')
print(f'Angulo de transmision maximo: {mu_max:.1f} grados')

---

## 15. Resumen y formulas clave

| Concepto | Formula / Definicion |
|----------|---------------------|
| **Maquina** | Solidos resistentes que transmiten fuerza/momento para realizar trabajo |
| **Mecanismo** | Cadena cinematica con un cuerpo fijo |
| **Eslabon** | Cada cuerpo con movimiento relativo |
| **Par cinematico** | Union entre dos eslabones con movimiento relativo |
| **Clase I ($g$)** | Restringe 2 GDL: rotacion, prismatico, rodadura sin deslizamiento |
| **Clase II ($h$)** | Restringe 1 GDL: leva (rodadura con deslizamiento) |
| **Par n-ario** | Equivale a $n - 1$ pares binarios |
| **Gruebler** | $G = 3(N-1) - 2g - h$ |
| **Grashof** | $s + l \leq p + q$ $\implies$ al menos un eslabon gira 360° |
| **Manivela-balancin** | Fijamos adyacente al mas corto (Grashof cumplido) |
| **Doble manivela** | Fijamos opuesto al mas corto (Grashof cumplido) |
| **Triple balancin** | Grashof NO cumplido |
| **Angulo transmision** | $\mu = \angle(\text{biela}, \text{salida})$. Optimo: $90°$. Minimo: $>40°$ |
| **Inversiones** | Fijar otro eslabon: $N$ inversiones posibles para $N$ eslabones |

### Errores frecuentes

1. **Olvidar contar la barra fija** como eslabon ($N$ incluye el bastidor)
2. **No descomponer pares ternarios** en $(n-1)$ pares binarios
3. **Confundir clase I y clase II:** la leva es clase II ($h$), no clase I
4. **Aplicar Grashof sin ordenar:** $s$ es siempre la **mas corta** y $l$ la **mas larga**
5. **Ignorar GDL pasivos:** rodillos en levas dan un GDL extra que no afecta al movimiento util